### Thoughts

#### I'm trying to organize the project the following way:

* **Phase 1** — Learn what's in the CSV. Look at the shape of the data. Find the Ghost Ticks. Understand what's messy about it. No cleaning yet — just detective work.
* **Phase 2** — Build the transform step. Sort by ticker and time. Decide on imputation strategy (forward fill vs. interpolation) and produce a Golden Record.
* **Phase 3** — Pull data from Yahoo Finance and/or FRED. Align it with the cleaned CSV. Deal with the fact that these sources have different granularity and formats. (I have some questions about this).
* **Phase 4** — Add quality flags. Show statistical evidence that the Golden Record is trustworthy.
* **Phase 5** — Run some actual analysis on the clean data — correlations between sectors, volatility measures, maybe a simple risk model.

#### Pre-analysis

I first took the data to Excel and did some quick data investigation, which helped me understand the data a little better. Here are my conclusions:

- Data goes from 9:30 to 9:35 (5 minutes of data).
- Each ETF has 880 entries (doing quick math, on average that's around 2.933 entries per second).
- My guess is that the data is synthetic, since I find it hard to believe that those ETFs are that volatile (need to prove that).

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:

# Read CSV file from project.
dataF = pd.read_csv("pedwar_raw_sector_ticks(in).csv")

#### Data Transformation

So far, the only thing I have done is set the day and time into a unified structure, as well as setting all the times to the same date since that made more sense to me.

In [ ]:

# Mask to separate formated dates of not fromated dates based "/"
dataTs = dataF['ts'].str.contains('/', na=False)

goodD = pd.to_datetime(dataF.loc[dataTs, 'ts']) 
badD = dataF.loc[~dataTs, 'ts'] # flag ~ tells just give me the bad ones 

In [ ]:

# Separating the bad dates or wrong formated dates to make it as the good dates
partsTs = badD.str.split(':')
minutesTs = partsTs.str[0].astype(int) # Minutes
secondsTs = partsTs.str[1].astype(float) # Seconds 

transformDates = pd.to_datetime({
    'year': 2026, 
    'month': 2,
    'day': 12,
    'hour': 9,
    'minute': minutesTs,
    'second': secondsTs,
    # 'microsecond': ((secondstr % 1) * 1_000_000).astype(int)
})

In [ ]:

dataF["timestamps"] = pd.NaT # Creating a new column 

# Populate the new column with all the formated values
dataF.loc[dataTs, "timestamps"] = goodD
dataF.loc[~dataTs, "timestamps"] = transformDates

# sample it, making sure is correct
sample = dataF[["ticker", "timestamps", "mid"]]
print(sample)

#### Pre-Analysis

I'm also trying to do some continuous analysis before I jump to the ghost ticks. From the following graph, you will see that an anomaly happens every minute across all ETFs.

In [ ]:

dataF = dataF.sort_values('timestamps') # Organize the data frame by time

# Graph stock movements to fins gaps in the data 
gstart = pd.Timestamp('2026-02-12 09:30:00')
gend = pd.Timestamp('2026-02-12 09:32:30')

gstart2 = pd.Timestamp('2026-02-12 09:32:30')
gend2 = pd.Timestamp('2026-02-12 09:35:00')

for stock in dataF["ticker"].unique():
    stock_data = dataF[dataF['ticker'] == stock]

    plt.figure(figsize=(12, 6))
    plt.plot(stock_data['timestamps'], stock_data['mid'], label = stock) # defining the X and Y axis 

    ax = plt.gca() # Get axis 
    ax.set_xlim(gstart, gend) # Defining limits 
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S')) # Format date to then: 
    ax.xaxis.set_major_locator(mdates.SecondLocator(interval = 15)) # Set Intervals of 15 seconds each 
    

    plt.xticks(rotation = 45)
    plt.title(f'{stock} Price Over Time') # each chart gets its stock name
    plt.style.use('bmh')
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.tight_layout()
    plt.show()

# --- Second chart ---
for stock in dataF["ticker"].unique():
    stock_data = dataF[dataF['ticker'] == stock]

    plt.figure(figsize=(12, 6))
    plt.plot(stock_data['timestamps'], stock_data['mid'], label = stock)

    ax = plt.gca()
    ax.set_xlim(gstart2, gend2)
    ax.xaxis.set_major_locator(mdates.SecondLocator(interval = 15))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))

    plt.xticks(rotation = 45)
    plt.title(f'{stock} Price Over Time') # each chart gets its stock name
    plt.style.use('bmh') # pre amde style by matplot lib 
    plt.xlabel('Date')
    plt.ylabel('Price')
    plt.tight_layout()
    plt.show()

In [ ]:

# General ETF's graphs 
plt.figure(figsize=(10, 6))

for stock in dataF["ticker"].unique():
    stock_data = dataF[dataF['ticker'] == stock]
    plt.plot(stock_data['timestamps'], stock_data['mid'], label=stock)

ax = plt.gca()
ax.set_xlim(gstart, gend)
ax.xaxis.set_major_locator(mdates.SecondLocator(interval=10))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.title('9:30:00 - 9:32:30')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.tight_layout()
plt.show()

# --- Second chart ---
plt.figure(figsize=(10, 6))

for stock in dataF["ticker"].unique():
    stock_data = dataF[dataF['ticker'] == stock]
    plt.plot(stock_data['timestamps'], stock_data['mid'], label=stock)

ax = plt.gca()
ax.set_xlim(gstart2, gend2)
ax.xaxis.set_major_locator(mdates.SecondLocator(interval=10))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
plt.xticks(rotation=45)
plt.title('9:32:30 - 9:35:00')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

# Analysis 
dataF["price_change"] = dataF.groupby('ticker')['mid'].diff().abs()
# print(dataF.groupby('ticker')['price_change'].describe())

# Getting the mean price by ETF taking into considerations all ticks
avg_all = dataF.groupby('ticker')['mid'].mean()
# print(avg_all)

# Getting the mean price by ETF taking into considerations all ticks but outliners
difference = 14.5
dataF['is_outlier'] = dataF["price_change"] > difference

avg = dataF.groupby(['ticker', 'is_outlier'])['mid'].mean()
#avg_time = dataF.groupby('timestamp')
print(avg)



FFT - 

In [ ]:
# Pick one stock
stock_name = dataF['ticker'].iloc[0]
one_stock = dataF[dataF['ticker'] == "XLB"]

# Show all the rows flagged as outliers
print(one_stock[one_stock['is_outlier']][['timestamps', 'mid', 'price_change']])

In [ ]:
dataF = dataF.sort_values(['ticker', 'timestamps']).reset_index(drop=True)

results = []

for stock in dataF['ticker'].unique():
    stock_data = dataF[dataF['ticker'] == stock]
    
    # Get just the outlier rows
    outliers = stock_data[stock_data['is_outlier']]
    
    for idx in outliers.index:
        # Get all rows BEFORE this outlier
        before = stock_data[stock_data.index < idx]
        
        # Keep only the normal ones
        normal_before = before[~before['is_outlier']]
        
        # Calculate the average
        if len(normal_before) > 0:
            avg_before = normal_before['mid'].mean()
            count = len(normal_before)
        else:
            avg_before = None
            count = 0
        
        results.append({
            'ticker': stock,
            'outlier_time': stock_data.loc[idx, 'timestamps'],
            'outlier_price': stock_data.loc[idx, 'mid'],
            'avg_before': avg_before,
            'normal_ticks_counted': count
        })

# Turn it into a table
results_table = pd.DataFrame(results)
print(results_table)
